# Soro-Kev-Omni data pipeline, first pass (2x T4)

**Kaggle settings:** GPU **T4 x2**, Internet on, ~40 GB disk. Every GPU stage runs one shard per GPU in parallel.
T4 has no bf16: teachers and generators load in fp16 (or 4-bit with fp16 compute).
Expected wall time on 2x T4: text 2 min, generation ~1.5 h, teacher ~2.5 h, TTS ~1 h, UI rows 10 min, audit + build 5 min.

| Stage | Model / source | VRAM per T4 | License |
| :-- | :-- | :-- | :-- |
| Contrastive generation | Qwen/Qwen3-4B-Instruct-2507, fp16 | ~9 GB | Apache-2.0 |
| Teacher scoring (4 rotations, one batched forward) | Qwen/Qwen3-8B, 4-bit NF4 | ~7 GB | Apache-2.0 |
| TTS | hexgrad/Kokoro-82M | <1 GB | Apache-2.0 |
| UI rows | OS-Atlas UIBert | CPU | Apache-2.0 repo, CC BY 4.0 upstream |
| Audio eval | RVtech/Audio2Tool (JSON only) | CPU | CC BY-NC 4.0, eval only |

Image-dependent routing needs the multimodal teacher (Qwen3-Omni-30B-A3B) on an A100 and runs separately.

In [ ]:
import os, subprocess, json, sys, glob
WORK = "/kaggle/working" if os.path.isdir("/kaggle") else "/content"
REPO = f"{WORK}/Soro-Boro"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "upp-p0", "https://github.com/nisaral/Soro-Boro.git", REPO], check=True)
os.chdir(REPO)
os.environ["PYTHONPATH"] = f"{REPO}/veyra/python/src"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
sys.path.insert(0, f"{REPO}/veyra/python/src")
import torch
NGPU = torch.cuda.device_count()
print("GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(NGPU)])

In [ ]:
# One package per install: a single failure (e.g. kokoro on Python 3.13) must not abort the rest.
import sys, subprocess, importlib
print("Python", sys.version.split()[0])
for pkg in ['transformers>=5.0', 'peft', 'bitsandbytes>=0.46.1', 'accelerate', 'huggingface_hub', 'soundfile',
            'kokoro-onnx>=0.4']:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", pkg], capture_output=True, text=True)
    print(("ok   " if r.returncode == 0 else "FAIL ") + pkg + ("" if r.returncode == 0 else " :: " + r.stderr.strip()[-300:]))
if sys.version_info < (3, 13):  # the kokoro package (PyTorch backend) supports up to 3.12
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kokoro>=0.9.2"])
importlib.invalidate_caches()
for mod in ("bitsandbytes", "transformers", "peft"):
    m = importlib.import_module(mod)
    print(mod, getattr(m, "__version__", "?"))
try:
    import kokoro_onnx  # noqa: F401
    print("TTS backend: kokoro-onnx available")
except ImportError as e:
    print("TTS backend missing:", e)

## 1. Text: convert, template-noise tagging, look-alikes, near-miss candidates, clean eval files (CPU)

In [ ]:
from huggingface_hub import snapshot_download
DATA = snapshot_download("nisaralll/soro-universal-tool-harness", repo_type="dataset")
!python veyra/omni_data/stage_text.py --data-dir {DATA} --out runs/omni_data/stage_text

## 2. Contrastive generation (RLCD-style), one shard per GPU

In [ ]:
%%bash
# Reuse requests generated in an earlier run if they are attached as a Kaggle dataset
# (e.g. /kaggle/input/omni-data-pass1/omni_data/gen/text_generated_raw.jsonl); otherwise generate.
PREV=$(ls /kaggle/input/*/omni_data/gen/text_generated_raw.jsonl 2>/dev/null | head -n 1)
mkdir -p runs/omni_data/gen
if [ -n "$PREV" ]; then
  echo "reusing $PREV"; cp "$PREV" runs/omni_data/gen/raw_0.jsonl
else
  N=$(python -c "import torch;print(max(1,torch.cuda.device_count()))")
  for k in $(seq 0 $((N-1))); do
    CUDA_VISIBLE_DEVICES=$k python veyra/omni_data/stages.py generate --in runs/omni_data/stage_text/text_base.jsonl --out runs/omni_data/gen/raw_$k.jsonl --model Qwen/Qwen3-4B-Instruct-2507 --limit 3000 --shard $k/$N > gen_$k.log 2>&1 &
  done
  wait
  tail -n 2 gen_*.log
fi

In [ ]:
!cat runs/omni_data/gen/raw_*.jsonl > runs/omni_data/gen/text_generated_raw.jsonl && wc -l runs/omni_data/gen/text_generated_raw.jsonl

## 3. Teacher scoring: near-miss candidates + generated rows + template-noise rows

In [ ]:
!mkdir -p runs/omni_data/labelled runs/omni_data/verified
!cat runs/omni_data/stage_text/text_near_miss.jsonl runs/omni_data/gen/text_generated_raw.jsonl runs/omni_data/stage_text/text_generic.jsonl > runs/omni_data/labelled/to_score.jsonl
!wc -l runs/omni_data/labelled/to_score.jsonl

In [ ]:
%%bash
N=$(python -c "import torch;print(max(1,torch.cuda.device_count()))")
for k in $(seq 0 $((N-1))); do
  CUDA_VISIBLE_DEVICES=$k python veyra/omni_data/stages.py teacher --model Qwen/Qwen3-8B --four-bit --in runs/omni_data/labelled/to_score.jsonl --out runs/omni_data/labelled/scored_$k.jsonl --shard $k/$N > teacher_$k.log 2>&1 &
done
wait
tail -n 2 teacher_*.log

In [ ]:
!cat runs/omni_data/labelled/scored_*.jsonl > runs/omni_data/labelled/scored.jsonl
!python veyra/omni_data/stages.py filter --in runs/omni_data/labelled/scored.jsonl --out runs/omni_data/labelled/verified_all.jsonl
from soro.omni_data.schema import read_rows, write_rows
v = list(read_rows("runs/omni_data/labelled/verified_all.jsonl"))
write_rows([r for r in v if "near_miss" in r.tags and "generated" not in r.tags], "runs/omni_data/verified/text_near_miss.jsonl")
write_rows([r for r in v if "generated" in r.tags], "runs/omni_data/verified/text_generated.jsonl")
write_rows([r for r in v if "generic_request" in r.tags], "runs/omni_data/verified/text_generic.jsonl")
print({k: sum(1 for _ in read_rows(f"runs/omni_data/verified/{k}.jsonl")) for k in ("text_near_miss", "text_generated", "text_generic")})

## 4. Audio: Kokoro TTS + noise / speed / reverb / telephone + streaming prefixes, one shard per GPU

In [ ]:
!cat runs/omni_data/stage_text/text_base.jsonl runs/omni_data/stage_text/text_collisions.jsonl runs/omni_data/verified/*.jsonl > runs/omni_data/tts_source.jsonl

In [ ]:
%%bash
N=$(python -c "import torch;print(max(1,torch.cuda.device_count()))")
for k in $(seq 0 $((N-1))); do
  CUDA_VISIBLE_DEVICES=$k python veyra/omni_data/stages.py tts --in runs/omni_data/tts_source.jsonl --out runs/omni_data/audio/tts_$k.jsonl --wav-dir runs/omni_data/audio/wav --limit 8000 --shard $k/$N > tts_$k.log 2>&1 &
done
wait
tail -n 2 tts_*.log

In [ ]:
!cat runs/omni_data/audio/tts_*.jsonl > runs/omni_data/audio/audio_tts.jsonl && wc -l runs/omni_data/audio/audio_tts.jsonl

## 5. UI images: OS-Atlas UIBert subset (CPU)

In [ ]:
from huggingface_hub import hf_hub_download
j = hf_hub_download("OS-Copilot/OS-Atlas-data", "mobile_domain/uibert_raw.json", repo_type="dataset")
z = hf_hub_download("OS-Copilot/OS-Atlas-data", "mobile_domain/UIBert.zip", repo_type="dataset")
!mkdir -p runs/omni_data/image/uibert && unzip -qo {z} -d runs/omni_data/image/uibert
recs = json.load(open(j))
recs = recs if isinstance(recs, list) else next(v for v in recs.values() if isinstance(v, list))
first = recs[0]["img_filename"]
cands = [d for d in glob.glob("runs/omni_data/image/uibert/**/", recursive=True) if os.path.exists(os.path.join(d, first))]
IMG_ROOT = cands[0] if cands else "runs/omni_data/image/uibert"
print("first record:", first, "-> image root:", IMG_ROOT)

In [ ]:
!python veyra/omni_data/stages.py osatlas --json {j} --images {IMG_ROOT} --subsource uibert --out runs/omni_data/image/ui_os_atlas.jsonl
from soro.omni_data.schema import read_rows
rows = list(read_rows("runs/omni_data/image/ui_os_atlas.jsonl"))
print(len(rows), "rows;", sum(os.path.exists(r.image) for r in rows[:500]), "of first 500 image paths exist")

## 6. Audio eval: Audio2Tool (CC BY-NC 4.0, eval only, JSON only)

In [ ]:
!python veyra/omni_data/stages.py a2t --out runs/omni_data/eval/eval_audio2tool.jsonl --root runs/omni_data/a2t --no-audio

## 7. Audit every file, then build (license gate, dedupe, leakage, verification, mix, splits)

In [ ]:
!python -m soro.omni_data.audit runs/omni_data/stage_text/*.jsonl runs/omni_data/verified/*.jsonl runs/omni_data/audio/audio_tts.jsonl runs/omni_data/image/ui_os_atlas.jsonl runs/omni_data/eval/*.jsonl --tokenizer Qwen/Qwen3-0.6B-Base --out runs/omni_data/audit
!python -m soro.omni_data.build --config veyra/omni_data/omni_mix.json --out runs/omni_data/build
rep = json.load(open("runs/omni_data/build/build_report.json"))
print(json.dumps({k: rep.get(k) for k in ("input_rows", "missing_inputs", "invalid", "dedupe", "leakage_before_filter", "dropped_unverified", "written")}, indent=1, default=str))
print(json.dumps(rep["mix"], indent=1))
print(json.dumps(rep["label_audit"], indent=1)[:1500])

In [ ]:
!cd runs && zip -qr ../omni_data_pass1.zip omni_data -x "omni_data/audio/wav/*" "omni_data/image/uibert/*" "omni_data/a2t/*"
print("Download omni_data_pass1.zip (rows, audit and build reports). Save the notebook output as a Kaggle dataset to keep audio and images.")